In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

import xgboost as xgb
from xgboost import plot_importance
import re
from pathlib import Path
import os

In [ ]:
INPUT = Path("/kaggle/input/store-sales-time-series-forecasting")
assert (INPUT/"train.csv").exists(), "train.csv not found under Kaggle input path"

train_path  = INPUT/"train.csv"
test_path   = INPUT/"test.csv"
oil_path    = INPUT/"oil.csv"
stores_path = INPUT/"stores.csv"
sub_path    = INPUT/"sample_submission.csv"

# --- Load ---
train  = pd.read_csv(train_path, parse_dates=['date'])
test   = pd.read_csv(test_path,  parse_dates=['date'])
oil    = pd.read_csv(oil_path,   parse_dates=['date'])
stores = pd.read_csv(stores_path)

# --- Merge (same as original) ---
train = train.merge(oil, on='date', how='left')
test  = test.merge(oil, on='date', how='left')
train = train.merge(stores, on='store_nbr', how='left')
test  = test.merge(stores, on='store_nbr', how='left')
print(train.shape, test.shape)

print(train.isnull().sum())
print(train.info())
print(train.nunique())
print(train.describe())
print(train.columns)
print(test.columns)

print(train['family'].nunique() * train['store_nbr'].nunique())
print(train['date'].nunique())
print(test.shape)

In [ ]:
# ---- Plotly EDA (same style) ----
import importlib.util, plotly.io as pio
pio.renderers.default = "png" if importlib.util.find_spec("kaleido") else "notebook_connected"

train_plot = train.groupby('family')['store_nbr'].count()
fig = px.bar(train_plot, y='store_nbr', title="Number of family with store_nbr")
fig.show()

for col in ['onpromotion', 'sales']:
    train_plot = train.groupby(['store_nbr', 'family'])[col].mean().reset_index()
    fig = px.line(train_plot, x='store_nbr', y=col, color='family',
                  title=f'Mean {col} on each store_nbr and family')
    fig.show()

for col in ['onpromotion', 'sales']:
    plot = train.groupby('date')[col].mean().reset_index()
    fig = px.line(plot, x='date', y=col, title=f'Total {col} per day')
    fig.show()

for col in ['sales', 'onpromotion']:
    for t in ['store_nbr', 'family']:
        plot = train.groupby(t)[col].mean()
        fig = px.bar(plot, y=col, title=f'Mean {col} in {t}')
        fig.show()

# ---- Datetime & calendar feats (same) ----
train['date'] = pd.to_datetime(train['date'], format='%Y-%m-%d')
test['date']  = pd.to_datetime(test['date'],  format='%Y-%m-%d')
train.date.dtype

train['year'] = train.date.dt.year
train['day_of_week'] = train.date.dt.dayofweek
train['month'] = train.date.dt.month

test['year'] = test.date.dt.year
test['day_of_week'] = test.date.dt.dayofweek
test['month'] = test.date.dt.month

for col in ['onpromotion', 'sales']:
    for t in ['day_of_week', 'month', 'year']:
        time = train.groupby(t)[col].mean()
        fig = px.bar(time, y=col, title=f'Mean {col} each {t}')
        fig.show()

# ---- Datetime & calendar feats (same) ----
train['date'] = pd.to_datetime(train['date'], format='%Y-%m-%d')
test['date']  = pd.to_datetime(test['date'],  format='%Y-%m-%d')
train.date.dtype

train['year'] = train.date.dt.year
train['day_of_week'] = train.date.dt.dayofweek
train['month'] = train.date.dt.month

test['year'] = test.date.dt.year
test['day_of_week'] = test.date.dt.dayofweek
test['month'] = test.date.dt.month

for col in ['onpromotion', 'sales']:
    for t in ['day_of_week', 'month', 'year']:
        time = train.groupby(t)[col].mean()
        fig = px.bar(time, y=col, title=f'Mean {col} each {t}')
        fig.show()

In [ ]:
# ---- Combine train/test for consistent encoding ----
data = pd.concat([train, test])
print(data.nunique())
print(data.isnull().sum())

# same ffill/bfill average for oil
data['dcoilwtico'] = (data['dcoilwtico'].fillna(method='ffill') +
                      data['dcoilwtico'].fillna(method='bfill')) / 2

data.fillna(0, inplace=True)
print(data.isnull().sum())

train_uniq = train[['id', 'date']]
test_uniq  = test[['id', 'date']]

train_sales = train[['sales']]

In [ ]:
# ---- Seaborn plots ----
plt.rcParams['figure.figsize'] = (12, 5)

fig, axes = plt.subplots(1, 2, figsize=(16,5))
sns.boxplot(data=train, x='day_of_week', y='sales', ax=axes[0])
axes[0].set_title('Whisker: Sales by Day of Week')
sns.boxplot(data=train, x='month', y='sales', ax=axes[1])
axes[1].set_title('Whisker: Sales by Month')
plt.show()

top_fams = train.groupby('family')['sales'].sum().sort_values(ascending=False).head(10).index
plt.figure(figsize=(14,6))
sns.boxplot(data=train[train['family'].isin(top_fams)], x='family', y='sales')
plt.title('Whisker: Sales by Top-10 Families')
plt.xticks(rotation=30)
plt.show()

num_cols = ['sales', 'onpromotion', 'dcoilwtico', 'year', 'month', 'day_of_week']
corr = train[num_cols].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix (numeric features)')
plt.show()


In [ ]:
# ---- Preprocess ----
data = data.drop(columns=['id', 'date', 'sales'])
data = pd.get_dummies(data, drop_first=True)

scaler = MinMaxScaler()
data = pd.DataFrame(scaler.fit_transform(data), columns=data.columns)

train_sales = pd.DataFrame(train_sales, columns=['sales'])
train_sales = pd.DataFrame(scaler.fit_transform(train_sales), columns=train_sales.columns)

print(data.describe())

In [ ]:
# Slice by exact counts
train_prep = data[:1782*1684]
train_sales = train_sales[:1782*1684]
test_prep  = data[1684*1782:]

In [ ]:
from packaging import version
import os
import xgboost as xgb

def has_gpu():
    try:
        import cupy
        return cupy.cuda.runtime.getDeviceCount() > 0
    except Exception:
        pass
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        pass
    return any(os.path.exists(p) for p in ("/dev/nvidia0", "/dev/nvidiactl")) or \
           bool(os.environ.get("CUDA_VISIBLE_DEVICES"))

use_gpu = has_gpu()
xgb_ver = version.parse(xgb.__version__)

common_params = dict(
    objective="count:poisson",
    booster="gbtree",
    n_estimators=200,
    max_depth=100,
    learning_rate=0.4,
    random_state=150,
)

if xgb_ver >= version.parse("2.0.0"):
    XGB_model = xgb.XGBRegressor(
        **common_params,
        tree_method="hist",
        device="cuda" if use_gpu else "cpu",
    )
else:
    XGB_model = xgb.XGBRegressor(
        **common_params,
        tree_method="gpu_hist" if use_gpu else "hist",
    )

print(f"XGBoost {xgb.__version__} | GPU used: {use_gpu}")

XGB_model.fit(train_prep, train_sales.values.ravel())


In [ ]:
# ---- Predict ----
pred_train = XGB_model.predict(train_prep)
pred_test  = XGB_model.predict(test_prep)

pred_train = pd.DataFrame(pred_train)
pred_train = scaler.inverse_transform(pred_train)
pred_train = np.array(pred_train)

train_sales_np = scaler.inverse_transform(pd.DataFrame(train_sales))
train_sales_np = np.array(train_sales_np)

pred_test = pd.DataFrame(pred_test)
pred_test = scaler.inverse_transform(pred_test)
pred_test = np.array(pred_test)

rmsle = np.sqrt(mean_squared_log_error(train_sales_np, pred_train))
print(f'RMSLE on real training set: {rmsle}')

pred_train = pd.DataFrame({'pred_train': pred_train.ravel()})
pred_test  = pd.DataFrame({'pred_train': pred_test.ravel()})

train['sales'].plot(kind='line', figsize=(15,5))
plt.show()

pred_train['pred_train'].plot(kind='line', figsize=(15,5))
plt.show()

pred_test['pred_train'].plot(kind='line', figsize=(15,5))
plt.show()

In [ ]:
# ---- Submission ----
submit = pd.read_csv(sub_path)
submit['sales'] = pred_test.values
display(submit.head())

out_path = Path("/kaggle/working/submission.csv")
submit.to_csv(out_path, index=False)
print(f"Saved: {out_path}")